# Agente Vitrinifarne — Notebook 2: índice e agente

Aqui o agente ganha vida. Ao final deste notebook você faz uma pergunta em português e
recebe uma resposta baseada nos documentos, com a fonte citada.

**O caminho:**

1. Ler os documentos (reaproveitando o módulo `src/leitores.py`)
2. Quebrar os textos em pedaços menores — os *chunks*
3. Transformar cada pedaço em vetor numérico — o *embedding*
4. Guardar os vetores num índice de busca — o *FAISS*
5. A cada pergunta: buscar os pedaços mais parecidos e mandar para o Gemini responder

**Pré-requisito:** chave de API do Google Gemini salva nos Secrets do Colab com o nome
`GOOGLE_API_KEY`.

---

## 1. Instalar as bibliotecas

Além das do notebook anterior, entram três novas:

| Biblioteca | Para quê |
|---|---|
| `langchain-text-splitters` | quebrar o texto em pedaços com sobreposição |
| `langchain-google-genai` | conversar com o Gemini (embeddings e respostas) |
| `faiss-cpu` | guardar os vetores e buscar por similaridade |

Leva cerca de um minuto.

In [ ]:
!pip install -q pypdf python-docx python-pptx openpyxl beautifulsoup4 pandas
!pip install -q langchain-text-splitters langchain-google-genai faiss-cpu
print("Pronto.")

## 2. A chave de API

**Nunca escreva a chave dentro do código.** Se você fizer isso e subir o notebook para o
GitHub, a chave fica pública — bots varrem o GitHub procurando exatamente isso, e o Google
revoga a chave em minutos.

O Colab tem um cofre para isso. No ícone de **chave** (🔑) na barra lateral esquerda:

1. Clique em *Adicionar novo secret*
2. Nome: `GOOGLE_API_KEY`
3. Valor: cole a chave que você gerou no Google AI Studio
4. Ative o botão *Acesso ao notebook*

A célula abaixo lê do cofre. A chave nunca aparece no código nem no resultado.

In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

chave = os.environ["GOOGLE_API_KEY"]
print(f"Chave carregada: {chave[:6]}...{chave[-4:]} ({len(chave)} caracteres)")

## 3. Descobrir qual modelo usar

Os nomes dos modelos do Gemini mudam com o tempo. Em vez de fixar um nome que pode estar
desatualizado, a célula abaixo testa alguns candidatos e fica com o primeiro que responder.

Se nenhum funcionar, o erro exibido já diz o motivo — geralmente chave inválida ou cota
esgotada.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

CANDIDATOS = ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-1.5-flash", "gemini-pro"]

MODELO = None
for nome in CANDIDATOS:
    try:
        teste = ChatGoogleGenerativeAI(model=nome, temperature=0)
        teste.invoke("responda apenas: ok")
        MODELO = nome
        print(f"Modelo disponível: {nome}")
        break
    except Exception as erro:
        print(f"  {nome} indisponível ({type(erro).__name__})")

if MODELO is None:
    raise RuntimeError("Nenhum modelo respondeu. Confira a chave de API.")

## 4. Trazer os documentos e o módulo de leitura

Clonamos o repositório e apontamos o Python para a pasta `src/`, onde mora o
`leitores.py`. É o mesmo código do Notebook 1, agora empacotado como módulo — assim ele
é escrito uma vez e usado em qualquer lugar.

In [ ]:
!rm -rf vitrinifarne-agente
!git clone -q https://github.com/francielleneves/vitrinifarne-agente.git

import sys, json
from pathlib import Path

RAIZ = Path("vitrinifarne-agente")
sys.path.append(str(RAIZ / "src"))

from leitores import extrair

PASTA_DOCS = RAIZ / "docs"
manifesto = json.loads((PASTA_DOCS / "manifesto.json").read_text(encoding="utf-8"))

base = []
for doc in manifesto["documentos"]:
    base.append({**doc, "texto": extrair(PASTA_DOCS / doc["arquivo"], doc["formato"])})

print(f"{len(base)} documentos lidos, {sum(len(d['texto']) for d in base)} caracteres.")

## 5. Quebrar em pedaços

Por que não mandar o documento inteiro para o Gemini? Dois motivos: custo (você paga por
texto enviado) e precisão (o modelo se perde em texto longo e a resposta fica genérica).

Então quebramos tudo em pedaços de mais ou menos 900 caracteres. Dois detalhes importam:

**A sobreposição de 150 caracteres.** Cada pedaço repete o final do anterior. Sem isso,
uma frase pode ser cortada ao meio e a informação se perde nas duas metades. Imagine o
corte caindo entre "o prazo de estorno é de" e "10 dias úteis" — nenhum dos dois pedaços
responderia à pergunta.

**A ordem dos separadores.** O `RecursiveCharacterTextSplitter` tenta cortar primeiro em
quebras de parágrafo, depois em linhas, depois em pontos finais. Só corta no meio de uma
palavra em último caso. Assim os pedaços respeitam a estrutura do texto.

Repare também que cada pedaço recebe um cabeçalho com o nome e a versão do documento.
Isso viaja junto para o Gemini e é o que permite a ele citar a fonte corretamente.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

divisor = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", "; ", " ", ""],
    length_function=len,
)

chunks = []
for doc in base:
    for i, pedaco in enumerate(divisor.split_text(doc["texto"]), start=1):
        chunks.append({
            "texto": f"[{doc['titulo']} — versão {doc['versao']}]\n{pedaco}",
            "doc_id": doc["id"],
            "titulo": doc["titulo"],
            "arquivo": doc["arquivo"],
            "versao": doc["versao"],
            "categoria": doc["categoria"],
            "parte": i,
        })

print(f"{len(chunks)} pedaços gerados.\n")

import pandas as pd
tabela = pd.DataFrame(chunks)
tabela["tamanho"] = tabela["texto"].str.len()
tabela.groupby(["doc_id", "titulo"]).agg(
    pedacos=("parte", "count"), tamanho_medio=("tamanho", "mean")
).round(0)

## 6. Transformar texto em números

Este é o conceito central do projeto.

Um *embedding* é uma lista de números que representa o significado de um texto. Textos com
sentido parecido geram listas de números parecidas — mesmo sem compartilhar nenhuma
palavra. É por isso que "quando meu dinheiro volta?" consegue encontrar um trecho que fala
em "prazo de estorno".

O modelo do Google devolve 768 números para cada pedaço. Nós normalizamos esses vetores
para que a comparação entre eles seja o cosseno do ângulo — um valor entre 0 e 1, onde 1
significa "praticamente o mesmo assunto".

Esta célula faz 60 chamadas à API e leva cerca de 30 segundos.

In [ ]:
import numpy as np
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

textos = [c["texto"] for c in chunks]
vetores = np.array(embeddings.embed_documents(textos), dtype="float32")

print("Formato da matriz:", vetores.shape, "→", vetores.shape[0],
      "pedaços de", vetores.shape[1], "números cada")
print("Primeiros 8 números do pedaço 1:", vetores[0][:8].round(3))

## 7. Montar o índice de busca

O FAISS guarda os vetores de um jeito que permite achar os mais parecidos rapidamente.

`IndexFlatIP` compara por produto interno. Como normalizamos os vetores antes, esse
produto interno é exatamente a similaridade por cosseno.

Para 60 pedaços isso é instantâneo. A mesma estrutura funcionaria com milhões — é a razão
de usar FAISS em vez de comparar um por um no Python.

In [ ]:
import faiss

faiss.normalize_L2(vetores)

indice = faiss.IndexFlatIP(vetores.shape[1])
indice.add(vetores)

print(f"Índice criado com {indice.ntotal} vetores de {indice.d} dimensões.")

## 8. A busca

A função abaixo recebe uma pergunta, converte em vetor pelo mesmo modelo e devolve os
pedaços mais parecidos, com a nota de similaridade.

Antes de partir para as respostas, vale rodar e olhar: se a busca traz o trecho errado,
não existe prompt que salve a resposta. **Quando o agente erra, o problema quase sempre
está aqui, não no modelo.**

In [ ]:
def buscar(pergunta, quantidade=5):
    """Devolve os pedaços mais parecidos com a pergunta."""
    vetor = np.array([embeddings.embed_query(pergunta)], dtype="float32")
    faiss.normalize_L2(vetor)
    notas, posicoes = indice.search(vetor, quantidade)

    resultados = []
    for nota, posicao in zip(notas[0], posicoes[0]):
        resultado = dict(chunks[posicao])
        resultado["similaridade"] = float(nota)
        resultados.append(resultado)
    return resultados


for r in buscar("quanto tempo demora para o dinheiro voltar?"):
    print(f"{r['similaridade']:.3f}  {r['doc_id']}  {r['titulo']} (parte {r['parte']})")

## 9. O prompt

Agora juntamos tudo: os trechos encontrados vão para o Gemini junto com a pergunta e um
conjunto de instruções.

As instruções não são enfeite. Cada linha resolve um problema real:

- **"Use apenas os trechos"** evita que o modelo invente a partir do que ele aprendeu na
  internet. Sem essa linha, ele responde sobre a política de devolução de outra loja
  qualquer e soa igualmente convincente.
- **"Cite o documento e a versão"** é o que transforma isso numa base de conhecimento
  corporativa, e não num chute bem escrito.
- **"Se não estiver nos trechos, diga que não encontrou"** é a instrução mais importante.
  Um agente que admite não saber é confiável. Um que inventa é pior que não ter agente.

In [ ]:
INSTRUCOES = """Você é o assistente interno da Vitrinifarne, uma loja online de casa e decoração.
Sua função é responder perguntas de colaboradores e clientes usando exclusivamente a
documentação oficial da empresa.

Regras:
1. Responda apenas com base nos trechos fornecidos abaixo. Não use conhecimento externo.
2. Sempre cite o documento e a versão de onde tirou a informação.
3. Se a resposta exigir combinar informações de documentos diferentes, faça a combinação
   e cite todas as fontes usadas.
4. Se a informação não estiver nos trechos, responda exatamente: "Não encontrei essa
   informação na documentação disponível." e sugira qual área procurar.
5. Responda em português do Brasil, de forma direta e objetiva. Sem saudações.
6. Valores em reais e prazos devem ser reproduzidos exatamente como aparecem nos trechos.

TRECHOS DA DOCUMENTAÇÃO:
{contexto}

PERGUNTA: {pergunta}

RESPOSTA:"""

modelo = ChatGoogleGenerativeAI(model=MODELO, temperature=0)


def responder(pergunta, quantidade=5, mostrar_fontes=True):
    """Busca os trechos relevantes e pede ao Gemini uma resposta baseada neles."""
    achados = buscar(pergunta, quantidade)

    contexto = "\n\n---\n\n".join(a["texto"] for a in achados)
    prompt = INSTRUCOES.format(contexto=contexto, pergunta=pergunta)

    resposta = modelo.invoke(prompt).content

    if mostrar_fontes:
        fontes = []
        for a in achados:
            marca = f"{a['titulo']} (v{a['versao']})"
            if marca not in fontes:
                fontes.append(marca)
        resposta += "\n\nTrechos consultados: " + "; ".join(fontes)

    return resposta


print("Agente pronto.")

## 10. Testar

Cinco perguntas escolhidas para exercitar coisas diferentes.

A quinta é a mais importante: **nenhum documento sozinho responde a ela**. É preciso o
catálogo (para saber que a estante é sob encomenda e quanto custa), a planilha de prazos
(região Norte) e o guia de entregas (o mínimo de frete grátis no Norte é diferente).
Se o agente acertar essa, ele está realmente cruzando fontes.

In [ ]:
PERGUNTAS = [
    "Qual o prazo para desistir de uma compra?",
    "Em quantas vezes posso parcelar e qual a parcela mínima?",
    "Qual o prazo de entrega expresso para o interior do Nordeste?",
    "Um cliente quer devolver um produto de higiene pessoal com o lacre aberto. Pode?",
    "Comprei uma estante modular para Belém. Quando chega e o frete é grátis?",
]

for i, pergunta in enumerate(PERGUNTAS, start=1):
    print("=" * 78)
    print(f"PERGUNTA {i}: {pergunta}")
    print("-" * 78)
    print(responder(pergunta))
    print()

## 11. Testar o que o agente NÃO sabe

Tão importante quanto acertar é recusar direito. Estas perguntas não têm resposta na
documentação: o agente deve dizer que não encontrou, e não inventar.

Se ele inventar aqui, aumente a clareza da regra 4 nas instruções ou reduza a quantidade
de trechos buscados.

In [ ]:
FORA_DO_ESCOPO = [
    "Qual o faturamento da empresa no último trimestre?",
    "Vocês entregam em Portugal?",
    "Quem é o CEO da Vitrinifarne?",
]

for pergunta in FORA_DO_ESCOPO:
    print("-" * 78)
    print("PERGUNTA:", pergunta)
    print(responder(pergunta, mostrar_fontes=False))
    print()

## 12. Salvar o índice

Gerar os embeddings custa tempo e cota de API. Salvando o índice, o notebook seguinte
(e o servidor na OCI) carrega tudo pronto, sem refazer as chamadas.

São dois arquivos: o `.faiss` com os vetores e o `.json` com os textos e metadados
correspondentes. Os dois precisam andar juntos — a posição no índice é o que liga um
ao outro.

In [ ]:
faiss.write_index(indice, "indice.faiss")

with open("chunks.json", "w", encoding="utf-8") as arquivo:
    json.dump(chunks, arquivo, ensure_ascii=False, indent=2)

for nome in ["indice.faiss", "chunks.json"]:
    print(f"{nome}: {Path(nome).stat().st_size / 1024:.1f} KB")

print("\nBaixe os dois pelo painel de arquivos à esquerda.")

---

## O que este notebook fez

Todo o diagrama da arquitetura, do começo ao fim:

| Etapa | Célula |
|---|---|
| Ler os 8 formatos | 4 |
| Quebrar em pedaços | 5 |
| Gerar embeddings | 6 |
| Montar o índice | 7 |
| Buscar por similaridade | 8 |
| Responder com o Gemini | 9 |

## Próximos passos

1. **Interface** — uma tela simples com Gradio, para não demonstrar o projeto no Colab
2. **Implantação na OCI** — subir a aplicação e deixar acessível por um link
3. **README** — arquitetura, exemplos de perguntas e respostas, instruções de execução